<a href="https://colab.research.google.com/github/kipkii/kipki-s-/blob/main/I_love_Stake_pipe_LIne.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time
import os
from datetime import datetime, timedelta

# ==========================================
# [설정 구역]
# ==========================================
# ★ 트래픽 5만 건 승인받은 '디코딩(Decoding)' 키를 넣으세요!
MY_SERVICE_KEY = "6b11791d7a259c09d100dd6a5ab6a1f2466da76d809269c8705232ac5d608fde"

ABATT_CODE = "0513"  # 0513: 전국 최대 규모 농협음성
BREED_CD = "024001"  # 024001: 한우

# 10년 치 전체 기간 설정 (기상청 데이터 기간과 일치시키세요)
START_DATE = "2016-01-01"
END_DATE = "2026-05-31"

OUTPUT_FILE = "BigWave_Cattle_Grade_10Years.csv"

# ==========================================
# [함수 구역] API 호출 및 데이터 추출
# ==========================================
def fetch_grade_data_daily(target_date, service_key):
    # 공식 문서 기반 최종 URL 세팅
    url = "http://data.ekape.or.kr/openapi-data/service/user/grade/auct/cattlePriceDetail"
    req_dt = target_date.strftime("%Y%m%d")

    # 문서 명세에 맞춘 파라미터 구성
    params = {
        "serviceKey": service_key,
        "abattCode": ABATT_CODE,
        "startYmd": req_dt,
        "endYmd": req_dt,
        "breedCd": BREED_CD,
        "defectIncludeYn": "Y"  # 결함 포함 가격 기준
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        raw_xml = response.text

        # 서버 에러 및 트래픽 제한 감지
        if "ERROR" in raw_xml or "LIMITED NUMBER" in raw_xml:
            print(f"\n⚠️ 서버 에러 메시지 감지: {raw_xml[:200]}")
            return "ERROR"

        root = ET.fromstring(raw_xml)
        data_list = []

        for item in root.findall(".//item"):
            def get_val(tag_name):
                node = item.find(tag_name)
                return node.text if node is not None and node.text else None

            # 🚨 공식 문서 기반 정확한 XML 태그 매핑
            data_list.append({
                "경매일자": req_dt,
                "시장코드": ABATT_CODE,
                "성별": get_val("judgeSexNm"),   # 문서 기준 '성별'
                "등급": get_val("gradeNm"),      # 문서 기준 '등급명'
                "경락두수": int(get_val("auctCnt")) if get_val("auctCnt") else 0,
                "평균도체중": float(get_val("weight")) if get_val("weight") else None,
                "평균가격": int(get_val("auctAmt")) if get_val("auctAmt") else None
            })

        return pd.DataFrame(data_list)

    except Exception as e:
        print(f"\n   ❌ 통신 에러: {e}")
        return None

# ==========================================
# [실행 구역] 일별 자동화 루프 (이어쓰기 기능 포함)
# ==========================================
if __name__ == "__main__":
    print(f"데이터 파이프라인 가동: {START_DATE} ~ {END_DATE} (트래픽 5만 건 확보!)\n")

    current_date = datetime.strptime(START_DATE, "%Y-%m-%d")
    end_date = datetime.strptime(END_DATE, "%Y-%m-%d")

    total_days = 0
    total_rows = 0

    while current_date <= end_date:
        print(f"📡 수집 중: {current_date.strftime('%Y-%m-%d')} ... ", end="")

        df_daily = fetch_grade_data_daily(current_date, MY_SERVICE_KEY)

        if type(df_daily) == str and df_daily == "ERROR":
            print("❌ 수집 중단: 서버 에러 발생")
            break

        elif df_daily is not None and not df_daily.empty:

            # 🚨 [업그레이드] 인코딩 에러(utf-8) 자동 감지 및 cp949 우회(Fallback) 로직
            if not os.path.exists(OUTPUT_FILE):
                try:
                    df_daily.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
                except UnicodeEncodeError:
                    print(" [utf-8 에러 감지 -> cp949로 생성] ", end="")
                    df_daily.to_csv(OUTPUT_FILE, index=False, encoding="cp949")
            else:
                try:
                    df_daily.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig", mode='a', header=False)
                except UnicodeEncodeError:
                    print(" [utf-8 에러 감지 -> cp949로 이어쓰기] ", end="")
                    df_daily.to_csv(OUTPUT_FILE, index=False, encoding="cp949", mode='a', header=False)

            rows_cnt = len(df_daily)
            total_rows += rows_cnt
            print(f"✅ 성공 ({rows_cnt}건)")

        else:
            print("⚠️ 경매 없음 (휴장일)")

        total_days += 1
        current_date += timedelta(days=1)

        # 서버 과부하 방지용 매너타임
        time.sleep(0.3)

    print("=" * 60)
    print(f"🎉 [수집 대성공] 총 {total_days}일 순회 완료, 누적 {total_rows}건의 알맹이 꽉 찬 데이터 저장 완료!")
    print(f"💾 최종 결과물: {OUTPUT_FILE}")

데이터 파이프라인 가동: 2016-01-01 ~ 2026-05-31 (트래픽 5만 건 확보!)

📡 수집 중: 2016-01-01 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-02 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-03 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-04 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-05 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-06 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-07 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-08 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-09 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-10 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-11 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-12 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-13 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-14 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-15 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-16 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-17 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-18 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-19 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-20 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-21 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-22 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-23 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-24 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-25 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-26 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-27 ... ✅ 성공 (27건)
📡 수집 중: 2016-01-28 ... ✅ 성공 

In [2]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time
import os

# ==========================================
# [설정 구역]
# ==========================================
# ★ 공공데이터포털(기상청)에서 발급받은 '디코딩(Decoding)' 키
MY_WTHR_KEY = "6b11791d7a259c09d100dd6a5ab6a1f2466da76d809269c8705232ac5d608fde"

# 💡 지점코드(stnIds) 꿀팁:
# 108은 '서울'입니다. 한우 데이터가 '농협음성' 기준이므로,
# 음성과 기후가 가장 비슷한 '충주(114)'나 '청주(131)'를 쓰시면 분석이 훨씬 정교해집니다!
STN_ID = "114"

# 수집 연도 설정 (한우 데이터와 동일한 2014 ~ 2023)
START_YEAR = 2016
END_YEAR = 2018

OUTPUT_FILE = "BigWave_Weather_10Years.csv"

# ==========================================
# [함수 구역] API 호출 및 데이터 추출
# ==========================================
def fetch_weather_data_yearly(year, service_key):
    url = "http://apis.data.go.kr/1360000/AsosDalyInfoService/getWthrDataList"

    # 해당 연도의 1월 1일부터 12월 31일까지 설정
    start_dt = f"{year}0101"
    end_dt = f"{year}1231"

    params = {
        "serviceKey": service_key,
        "numOfRows": 400,       # 1년은 최대 366일이므로 400으로 넉넉하게 설정
        "pageNo": 1,
        "dataCd": "ASOS",
        "dateCd": "DAY",
        "startDt": start_dt,
        "endDt": end_dt,
        "stnIds": STN_ID
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        raw_xml = response.text

        root = ET.fromstring(raw_xml)
        data_list = []

        for item in root.findall(".//item"):
            def get_val(tag_name):
                node = item.find(tag_name)
                return node.text if node is not None and node.text else None

            # 폭염 및 THI(열스트레스) 지수 계산에 필수적인 핵심 피처만 추출
            data_list.append({
                "일자": get_val("tm"),
                "지점코드": get_val("stnId"),
                "평균기온": float(get_val("avgTa")) if get_val("avgTa") else None,
                "최고기온": float(get_val("maxTa")) if get_val("maxTa") else None,
                "최저기온": float(get_val("minTa")) if get_val("minTa") else None,
                "일강수량": float(get_val("sumRn")) if get_val("sumRn") else 0.0,
                "평균상대습도": float(get_val("avgRhm")) if get_val("avgRhm") else None
            })

        return pd.DataFrame(data_list)

    except Exception as e:
        print(f"\n   ❌ {year}년 통신 에러: {e}")
        return None

# ==========================================
# [실행 구역] 연도별 순회 수집
# ==========================================
if __name__ == "__main__":
    print(f"⛅ Big Wave 기상 데이터 수집 가동: {START_YEAR}년 ~ {END_YEAR}년\n")

    total_rows = 0

    for year in range(START_YEAR, END_YEAR + 1):
        print(f"📡 {year}년 데이터 뭉치 수집 중... ", end="")

        df_year = fetch_weather_data_yearly(year, MY_WTHR_KEY)

        if df_year is not None and not df_year.empty:
            # CSV 실시간 이어쓰기 및 인코딩 방어막 (cp949 Fallback)
            if not os.path.exists(OUTPUT_FILE):
                try:
                    df_year.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
                except UnicodeEncodeError:
                    df_year.to_csv(OUTPUT_FILE, index=False, encoding="cp949")
            else:
                try:
                    df_year.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig", mode='a', header=False)
                except UnicodeEncodeError:
                    df_year.to_csv(OUTPUT_FILE, index=False, encoding="cp949", mode='a', header=False)

            rows_cnt = len(df_year)
            total_rows += rows_cnt
            print(f"✅ 성공 ({rows_cnt}일치 저장완료)")

        else:
            print("⚠️ 데이터 없음 또는 통신 실패")

        # 기상청 서버 매너타임
        time.sleep(1)

    print("=" * 60)
    print(f"🎉 [수집 대성공] 10년 치 총 {total_rows}일의 기상 데이터가 완벽하게 저장되었습니다!")
    print(f"💾 최종 결과물: {OUTPUT_FILE}")

⛅ Big Wave 기상 데이터 수집 가동: 2016년 ~ 2018년

📡 2016년 데이터 뭉치 수집 중... ✅ 성공 (366일치 저장완료)
📡 2017년 데이터 뭉치 수집 중... ✅ 성공 (365일치 저장완료)
📡 2018년 데이터 뭉치 수집 중... ✅ 성공 (365일치 저장완료)
🎉 [수집 대성공] 10년 치 총 1096일의 기상 데이터가 완벽하게 저장되었습니다!
💾 최종 결과물: BigWave_Weather_10Years.csv
